In [ ]:
# ==============================================================================
# 1. SPARK INITIALIZATION & ENVIRONMENT SETUP
# ==============================================================================
!pip install pyspark -q

import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_unixtime, to_timestamp, lag, avg, stddev, min, max
from pyspark.sql.window import Window
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.regression import LinearRegression, GBTRegressor
from pyspark.ml.evaluation import RegressionEvaluator

# Initialize Spark Session configured for time-series memory demands
spark = SparkSession.builder \
    .appName("Bitcoin_Time_Series_Forecasting") \
    .config("spark.driver.memory", "4g") \
    .master("local[*]") \
    .getOrCreate()

print("Spark Session Successfully Initialized!")

Spark Session Successfully Initialized!


In [ ]:
# ==============================================================================
# 2. DATA LOADING & PREPROCESSING
# ==============================================================================
# Note: Ensure bitcoindirectory or kaggle 'bitstampUSD_1-min_data_2012-01-01_to_2021-03-31.csv' is uploaded
file_path = "bitstampUSD_1-min_data_2012-01-01_to_2021-03-31.csv"

# Load CSV dataset into Spark DataFrame
raw_df = spark.read.csv(file_path, header=True, inferSchema=True)

# Clean missing values & format Timestamp to readable DateTime
clean_df = raw_df.dropna() \
    .withColumn("Timestamp_dt", to_timestamp(from_unixtime(col("Timestamp")))) \
    .orderBy("Timestamp")

print(f"Total Processed Records: {clean_df.count()}")
clean_df.select("Timestamp", "Timestamp_dt", "Open", "High", "Low", "Close", "Volume").show(5)

Total Processed Records: 93440
+----------+-------------------+----+----+----+-----+------+
| Timestamp|       Timestamp_dt|Open|High| Low|Close|Volume|
+----------+-------------------+----+----+----+-----+------+
|1325376060|2012-01-01 00:01:00|4.58|4.58|4.58| 4.58|   0.0|
|1325376120|2012-01-01 00:02:00|4.58|4.58|4.58| 4.58|   0.0|
|1325376180|2012-01-01 00:03:00|4.58|4.58|4.58| 4.58|   0.0|
|1325376240|2012-01-01 00:04:00|4.58|4.58|4.58| 4.58|   0.0|
|1325376300|2012-01-01 00:05:00|4.58|4.58|4.58| 4.58|   0.0|
+----------+-------------------+----+----+----+-----+------+
only showing top 5 rows


In [ ]:
# ==============================================================================
# 3. TEMPORAL FEATURE ENGINEERING & TARGET FORMULATION
# ==============================================================================
# Define chronological window ordered by timestamp
time_window = Window.orderBy("Timestamp")

# 1. Lag Features: Historical close prices (t-1, t-2, t-3)
df_featured = clean_df \
    .withColumn("lag_1", lag("Close", 1).over(time_window)) \
    .withColumn("lag_2", lag("Close", 2).over(time_window)) \
    .withColumn("lag_3", lag("Close", 3).over(time_window))

# 2. Rolling Window Features: Moving averages & statistics over past windows
window_5m = Window.orderBy("Timestamp").rowsBetween(-5, -1)
window_15m = Window.orderBy("Timestamp").rowsBetween(-15, -1)

df_featured = df_featured \
    .withColumn("rolling_avg_5", avg("Close").over(window_5m)) \
    .withColumn("rolling_avg_15", avg("Close").over(window_15m)) \
    .withColumn("rolling_std_15", stddev("Close").over(window_15m))

# 3. Target Variable: Future Close Price (t+1 horizon prediction)
df_featured = df_featured.withColumn("target_future_close", lag("Close", -1).over(time_window))

# Drop null values created by lag/rolling offsets
final_df = df_featured.dropna()

In [ ]:
# ==============================================================================
# 4. FEATURE VECTOR ASSEMBLY & SCALING
# ==============================================================================
feature_cols = [
    "Open", "High", "Low", "Close", "Volume",
    "lag_1", "lag_2", "lag_3", "rolling_avg_5", "rolling_avg_15", "rolling_std_15"
]

assembler = VectorAssembler(inputCols=feature_cols, outputCol="raw_features")
assembled_df = assembler.transform(final_df)

scaler = StandardScaler(inputCol="raw_features", outputCol="features", withStd=True, withMean=True)
scaler_model = scaler.fit(assembled_df)
scaled_df = scaler_model.transform(assembled_df)

In [ ]:
# ==============================================================================
# 5. CHRONOLOGICAL TIME-BASED TRAIN-TEST SPLIT (No Data Leakage)
# ==============================================================================
# 80% Train, 20% Test split based on chronological timestamp cutoff
split_ratio = 0.80
total_count = scaled_df.count()
train_count = int(total_count * split_ratio)

# Retrieve cutoff timestamp for deterministic temporal split
cutoff_row = scaled_df.select("Timestamp").orderBy("Timestamp").collect()[train_count]
cutoff_timestamp = cutoff_row["Timestamp"]

train_df = scaled_df.filter(col("Timestamp") < cutoff_timestamp).cache()
test_df = scaled_df.filter(col("Timestamp") >= cutoff_timestamp).cache()

print(f"Training Set Count (Historical): {train_df.count()}")
print(f"Testing Set Count (Future): {test_df.count()}")

Training Set Count (Historical): 74748
Testing Set Count (Future): 18688


In [ ]:
# ==============================================================================
# 6. MODEL TRAINING (LINEAR REGRESSION & GBT REGRESSOR)
# ==============================================================================
# Model 1: Distributed Linear Regression
lr = LinearRegression(featuresCol="features", labelCol="target_future_close")
lr_model = lr.fit(train_df)
lr_predictions = lr_model.transform(test_df)

# Model 2: Gradient Boosted Trees (GBT) Regressor
gbt = GBTRegressor(featuresCol="features", labelCol="target_future_close", maxDepth=5, maxIter=20, seed=42)
gbt_model = gbt.fit(train_df)
gbt_predictions = gbt_model.transform(test_df)

In [ ]:
# ==============================================================================
# 7. MODEL EVALUATION & COMPARISON METRICS
# ==============================================================================
# Define evaluation metrics for regression
evaluator_rmse = RegressionEvaluator(labelCol="target_future_close", predictionCol="prediction", metricName="rmse")
evaluator_mae = RegressionEvaluator(labelCol="target_future_close", predictionCol="prediction", metricName="mae")
evaluator_r2 = RegressionEvaluator(labelCol="target_future_close", predictionCol="prediction", metricName="r2")

# Evaluate Linear Regression Model
lr_rmse = evaluator_rmse.evaluate(lr_predictions)
lr_mae = evaluator_mae.evaluate(lr_predictions)
lr_r2 = evaluator_r2.evaluate(lr_predictions)

print(f"Linear Regression - RMSE: {lr_rmse:.3f}")
print(f"Linear Regression - MAE: {lr_mae:.3f}")
print(f"Linear Regression - R^2: {lr_r2:.3f}")

# Evaluate GBT Regressor Model
gbt_rmse = evaluator_rmse.evaluate(gbt_predictions)
gbt_mae = evaluator_mae.evaluate(gbt_predictions)
gbt_r2 = evaluator_r2.evaluate(gbt_predictions)

print(f"GBT Regressor - RMSE: {gbt_rmse:.3f}")
print(f"GBT Regressor - MAE: {gbt_mae:.3f}")
print(f"GBT Regressor - R^2: {gbt_r2:.3f}")

Linear Regression - RMSE: 0.008
Linear Regression - MAE: 0.001
Linear Regression - R^2: 0.998
GBT Regressor - RMSE: 0.085
GBT Regressor - MAE: 0.058
GBT Regressor - R^2: 0.798
